# Generate manuscript figures

Run the rank-test programs first to generate `random_summary.csv` and `all_pair_extension.csv`. Generates the vector PDF plots under `figures/`. Run from the repository root.

Run the cells in order using the Python 3 kernel in the repository root. Dependencies are listed in `requirements.txt`.

## Plotting setup

In [ ]:
"""Generate all vector figures from analytical formulas and illustrative graphs."""
from pathlib import Path
import math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, FancyArrowPatch
from scipy.optimize import brentq
D=Path.cwd()/'figures'; D.mkdir(exist_ok=True)
plt.rcParams.update({'font.family':'serif','font.size':10,'mathtext.fontset':'stix','axes.labelsize':11,'axes.spines.top':False,'axes.spines.right':False,'pdf.fonttype':42,'axes.linewidth':.8,'xtick.direction':'out','ytick.direction':'out','savefig.bbox':'tight','savefig.pad_inches':.04})
blue='#266590'; teal='#2b806c'; orange='#b97534'; grey='#e9edf0'; dark='#243647'; c=1/np.sqrt(2)
def save(fig,name): fig.savefig(D/(name+'.pdf')); plt.close(fig)


## Activation boundaries

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(7,3.15),layout='constrained')
x=np.linspace(0,1,700); y=np.sqrt(1-x*x)
ax[0].fill_between(x,0,y,color=grey); ax[0].fill_between(x,y,1,color=teal,alpha=.22); ax[0].plot(x,y,color=teal,lw=1.8)
ax[0].text(.24,.24,'Compatible\n(no GMS)',ha='center',color=dark); ax[0].text(.73,.94,'GMS',ha='center',color=teal,weight='bold'); ax[0].text(.51,.73,r'$\eta_0^2+\eta_1^2=1$',rotation=-34,ha='center',va='center',fontsize=9,bbox=dict(fc='white',ec='none',alpha=.8,pad=1)); ax[0].set(xlabel=r'Setting sharpness $\eta_0$',ylabel=r'Setting sharpness $\eta_1$',title='(a) One untrusted qubit')
ax[1].axhspan(0,1,fc=grey); ax[1].fill_between(x, c,1,where=x>=c,color=orange,alpha=.27); xx=np.linspace(c,1,400); ax[1].fill_between(xx,c/xx,1,color=teal,alpha=.42); ax[1].plot(xx,c/xx,color=teal,lw=1.8); ax[1].axvline(c,color=orange,ls='--',lw=.9); ax[1].axhline(c,color=orange,ls='--',lw=.9); t=2**(-.25); ax[1].plot(t,t,'o',color=dark,ms=4); ax[1].annotate(r'$(t,t)$',(t,t),(.60,.91),arrowprops={'arrowstyle':'-','lw':.7},fontsize=9); ax[1].plot(.75,.75,'o',color=orange,ms=4); ax[1].annotate('Both incompatible;\nno GMS',(.75,.75),(.27,.84),ha='center',va='center',arrowprops={'arrowstyle':'-','lw':.7},fontsize=8); ax[1].text(.30,.29,'Universally\nnon-GMS',ha='center',color=dark); ax[1].text(.93,.96,'GMS\npossible',ha='center',va='top',fontsize=8.4,color=dark); ax[1].set(xlabel=r'Device sharpness $\eta_A$',ylabel=r'Device sharpness $\eta_B$',title='(b) Two untrusted qubits')
for a in ax: a.set(xlim=(0,1),ylim=(0,1),aspect='equal',xticks=[0,.5,c,1],yticks=[0,.5,c,1]); a.set_xticklabels(['0','0.5',r'$1/\sqrt{2}$','1']); a.set_yticklabels(['0','0.5',r'$1/\sqrt{2}$','1']);
save(fig,'activation')


## Graph-state schematics

In [ ]:
# graph families, schematic only
fig,axs=plt.subplots(1,3,figsize=(7.2,2.35),layout='constrained')
def graph(ax,pos,edges,untrusted,title,labels=None):
 for u,v in edges: ax.plot([pos[u][0],pos[v][0]],[pos[u][1],pos[v][1]],color='#738590',lw=1.5,zorder=1)
 for u,(x,y) in enumerate(pos):
  ax.scatter(x,y,s=260,c=orange if u in untrusted else blue,edgecolors='white',linewidths=1.2,zorder=3)
  if labels is not None: ax.text(x,y,labels[u],ha='center',va='center',color='white',fontsize=9,zorder=4)
 ax.set_title(title,fontsize=10,pad=5); ax.set_aspect('equal'); ax.axis('off'); ax.margins(.21)
graph(axs[0],[(0,0),(1,0),(2,.6),(2,-.6),(3,.6),(3,-.6)],[(0,1),(1,2),(1,3),(2,3),(2,4),(3,5),(4,5)],{0,1},'(a) Leaf graph',['$\\ell$','$p$','','','',''])
N=7; pos=[(np.cos(2*np.pi*i/N+np.pi/2),np.sin(2*np.pi*i/N+np.pi/2)) for i in range(N)]; graph(axs[1],pos,[(i,(i+1)%N) for i in range(N)],{0,1},'(b) Cycle graph',[str(i) for i in range(N)])
rng=np.random.default_rng(15); N=9; pos=[(np.cos(2*np.pi*i/N+np.pi/2),np.sin(2*np.pi*i/N+np.pi/2)) for i in range(N)]; edges=[(i,j) for i in range(N) for j in range(i+1,N) if rng.random()<.5]; graph(axs[2],pos,edges,{0,4},'(c) Dense graph',['$a$','','','','$b$','','','',''])
save(fig,'graphs')


## Analytical failure-probability bound

In [ ]:
# analytical union bound
ns=np.arange(8,71)
def bound(n):
 m=n-2; sm=sum(math.comb(m,s)*(2**(s+2)-1)*2.**(-(m-s)) for s in (1,2)); lg=sum(math.comb(m,s)*2.**(2*s-(s-2)*(m-s))/6 for s in range(3,m//2+1)); return min(1,math.comb(n,2)*(3*2.**(-m)+sm+lg))
fig,ax=plt.subplots(figsize=(3.45,2.5),layout='constrained'); ax.semilogy(ns,[bound(n) for n in ns],color=blue,lw=2); ax.set(xlabel='Number of vertices $n$',ylabel='Failure probability bound',ylim=(1e-12,1.6)); ax.grid(alpha=.18,which='major'); ax.text(38,2e-3,'Analytical\nunion bound',fontsize=9,color=blue); save(fig,'random')


## White noise and local depolarization

In [ ]:
# noise boundaries, tight projected witness, and penalty comparator
u=np.linspace(c,1,400); g=1-c/u
p=[brentq(lambda z: .5*k*((1-z)**3+4*(1-z)**4+3*(1-z)**5)-2*np.sqrt(2),0,.2) if k>c else 0 for k in u]
pen=4-2*np.sqrt(2); gpen=1-(2*np.sqrt(2)+pen)/(4*u+pen)
fig,axs=plt.subplots(1,2,figsize=(7,2.8),layout='constrained')
for ax,y,ylab,title,ymax in zip(axs,[g,p],['Global white-noise fraction $1-v$','Local depolarization $p$'],['(a) Global noise','(b) Independent local noise'],[.325,.09]):
 ax.fill_between(u,0,y,color=teal,alpha=.22); ax.plot(u,y,color=teal,lw=2,label='Projected (8 terms)'); ax.axvline(c,color=blue,ls='--',lw=1); ax.text(.90,ymax*.22,'GMS certified',color=teal,ha='center',fontsize=9); ax.plot(1,y[-1],'o',ms=4,color=teal); ax.set(xlim=(.67,1.015),ylim=(0,ymax),xlabel=r'Sharpness product $u=\eta_A\eta_B$',ylabel=ylab,title=title,xticks=[c,.8,.9,1]); ax.set_xticklabels([r'$c$','0.8','0.9','1']); ax.annotate(f'{100*y[-1]:.2f}%',(1,y[-1]),(.96,y[-1]*.83),fontsize=8,ha='center',arrowprops=dict(arrowstyle='-',lw=.6))
axs[0].plot(u,gpen,color=blue,lw=1.3,ls='-.',label='Penalty (5 terms)'); axs[0].legend(loc='upper left',fontsize=8,frameon=False)
save(fig,'noise')


## Finite-size frequencies

In [ ]:
# finite-sample frequencies with exact cut-rank computations, Wilson intervals
import csv
rows=list(csv.DictReader(open(Path.cwd()/'random_summary.csv')))
def wilson(k,n):
 p=k/n; z=1.95996398454; d=1+z*z/n; a=(p+z*z/(2*n))/d; w=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d; return p, max(0,a-w),min(1,a+w)
fig,ax=plt.subplots(figsize=(3.45,2.65),layout='constrained')
for key,den,col,marker,label in [('fixed_pair_pass','samples',blue,'o','Designated pair'),('all_pair_pass','all_pair_samples',orange,'s','Every pair')]:
 data=[(int(r['n']),*wilson(int(r[key]),int(r[den]))) for r in rows if int(r[den])>0]; x,y,lo,hi=np.array(data).T
 ax.errorbar(x,y,yerr=[np.maximum(0,y-lo),np.maximum(0,hi-y)],fmt=marker+'-',color=col,markersize=3.2,lw=1,capsize=2,label=label)
ax.set(xlabel='Number of vertices $n$',ylabel='Certificate frequency',ylim=(-.04,1.05),xticks=[6,8,10,12,14,16,18,20]); ax.legend(loc='upper left',fontsize=8,frameon=False);ax.grid(alpha=.16);save(fig,'finite')


## Extended all-pair dataset

In [ ]:
# extended all-pair dataset: independent samples, combined with original counts
ext=list(csv.DictReader(open(Path.cwd()/'all_pair_extension.csv')))
fig,ax=plt.subplots(figsize=(3.45,2.65),layout='constrained')
for key,den,col,marker,label in [('fixed_pair_pass','samples',blue,'o','Designated pair'),('all_pair_pass','all_pair_samples',orange,'s','Every pair')]:
 data=[(int(r['n']),*wilson(int(r[key]),int(r[den]))) for r in rows if int(r[den])>0]
 if key=='all_pair_pass':
  for n in sorted({int(r['n']) for r in ext}):
   a=[int(r['all_pair_pass']) for r in ext if int(r['n'])==n];data.append((n,*wilson(sum(a),len(a))))
 x,y,lo,hi=np.array(data).T
 ax.errorbar(x,y,yerr=[np.maximum(0,y-lo),np.maximum(0,hi-y)],fmt=marker+'-',color=col,markersize=3.2,lw=1,capsize=2,label=label)
ax.set(xlabel='Number of vertices $n$',ylabel='Certificate frequency',ylim=(-.04,1.05),xticks=np.arange(6,25,3));ax.legend(loc='upper left',fontsize=8,frameon=False);ax.grid(alpha=.16);save(fig,'finite')


## Gate/readout and matched shot-budget comparison

In [ ]:
# gate/readout and power budget comparison, same independent-shot estimator
# Load the model definitions from the first code cell of the companion notebook.
import nbformat
_noise_nb = nbformat.read(Path.cwd() / "noise_budget.ipynb", as_version=4)
_noise_source = next(cell.source for cell in _noise_nb.cells if cell.cell_type == "code")
_noise_ns = {"__name__": "noise_budget"}
exec(compile(_noise_source, "noise_budget.ipynb", "exec"), _noise_ns)
scores, bounds, weights = (_noise_ns[k] for k in ("scores", "bounds", "weights"))
q=np.linspace(0,.065,400); vals=np.array([scores(z,.01) for z in q]);margin=vals-np.array(bounds)
fig,ax=plt.subplots(1,2,figsize=(7.0,2.65),layout='constrained')
for i,(label,col,style) in enumerate(zip(['Generator','Penalty','Projected'],[orange,blue,teal],['--','-.','-'])):
 ax[0].plot(100*q,margin[:,i],label=label,color=col,ls=style,lw=1.6)
 pos=margin[:,i]>0;N=8*np.log(100)*weights[i]**2/margin[pos,i]**2
 ax[1].semilogy(100*q[pos],N,label=label,color=col,ls=style,lw=1.6)
ax[0].axhline(0,color='0.4',lw=.7);ax[0].set(xlabel='Two-qubit depolarization $q$ (%)',ylabel='Certification margin $F-B_F$',title='(a) Expected violation');ax[1].set(xlabel='Two-qubit depolarization $q$ (%)',ylabel='Sufficient total shots',title='(b) Matched confidence and power',ylim=(1e2,1e7));ax[0].legend(frameon=False,fontsize=8);ax[1].grid(alpha=.15,which='both');ax[0].grid(alpha=.15);save(fig,'gatebudget')
